# មេរៀនទី ១៨ (បន្ត): បង្កាន់ដៃដែលបញ្ជាក់ថា *មនុស្ស* បានអនុម័តសកម្មភាព

មេរៀននេះបញ្ជាក់អំពីអ្វីដែល **ភ្នាក់ងារ** បានធ្វើ និងអ្វីដែល **ទ្វារ** បានសម្រេច។ សៀវភៅកំណត់ត្រានេះបន្ថែមផ្នែកខ្វះគ្នា៖ បញ្ជាក់ថា **មនុស្សមានឈ្មោះ** បានអនុម័តសកម្មភាព **ច្បាស់លាស់** — ហត្ថលេខា​ដែលមានជាផ្សេងពីគ្នា និងកាន់ដោយមនុស្សលើសកម្មភាពបែបប្រព័ន្ធពេញលេញ ដែលត្រូវបានបញ្ជាក់ក្រៅបណ្ដាញ។

ឯកសារទាំងពីរនៅទីនេះប្រើប្លង់ស្តាំ​ដូចគ្នានឹង **ប្លង់សំបុត្ររបស់មេរៀន**៖ ទម្រង់ផ្ទុកទិន្នន័យត្រង់មានវាល `type` ដែលត្រូវបានហត្ថលេខារួចដោយ Ed25519 លើប៊ីត JCS បែបប្រព័ន្ធដោយផ្ទាល់ ជាមួយវត្ថុ `signature` ដែលមានរចនាសម្ព័ន្ធភ្ជាប់ក៏ដូចជាត្រូវបានដកចេញពីប៊ីតដែលបានហត្ថលេខា។ សំបុត្រអនុម័តគឺជាប្រភេទថ្មីមួយ (`human.approval.v1`) រួមជាមួយប្រភេទសកម្មភាព ដូច្នេះ `verify_chain` មួយអាចគ្របដណ្ដប់ឯកសារពីរប្រភេទជាមួយប្លង់កូដដដែលដែលអ្នកបានបង្កើតនៅក្នុងសៀវភៅកំណត់ត្រាធំ។ សំបុត្រអនុម័តដោយមនុស្សនេះគឺជា​សមាសភាព​អប់រំ​ដែលកំណត់នៅទីនេះ មិនមែនជាប្រភេទសំបុត្រដែលកំណត់ដោយ draft-farley-acta-signed-receipts ទេ។

ការឡើងកម្រិតមួយដែលមានគោលបំណងលើកម្មវិធីផ្ទៀងផ្ទាត់ក្នុងសៀវភៅកំណត់ត្រាធំ៖ កម្មវិធីផ្ទៀងផ្ទាត់នៅទីនេះដោះស្រាយ `signature.key_id` ទៅនឹង **បញ្ជីកូនសោភ្ជាប់តាំងពីដើម** ជំនួសការជឿទុកចិត្តលើកូនសោសាធារណៈដែលមានក្នុងសំបុត្រ។ នេះគឺជាវិស័យផលិតកម្មដែលបញ្ជីត្រួតពិនិត្យរបស់មេរៀនផ្តល់អនុសាសន៍ ("ផ្សព្វផ្សាយកូនសោសាធារណៈសម្រាប់ផ្ទៀងផ្ទាត់") ហើយវាជា្វសារចម្បងដែលធ្វើឱ្យការចម្លងនោះបដិសេធបានជំនួសការចោលបញ្ជីកូនសោ។

ច្បាប់ដែលសៀវភៅកំណត់ត្រានេះបង្រៀន៖ **ការអនុម័តដែលបានហត្ថលេខា មិនមែនជាអំណាចដោយខ្លួនឯងទេ។** អំណាចមានតែបើសំបុត្រអនុម័ត និង សំបុត្រ​សកម្មភាព នៅតែភ្ជាប់ទៅនឹងសកម្មភាពប្រព័ន្ធដូចគ្នានៅពេលអនុវត្ត ក្រោមកំណែគោលនយោបាយ កូនសោ និងកំណត់ពេលធ្វើការដែលនៅតែមានសុពលភាព ហើយការអនុម័តនោះមិនទាន់បានប្រើប្រាស់រួចហើយ។ ការបរាជ័យមួយៗបដិសេធជាមួយ **ហេតុផលខុសៗគ្នា** ដូច្នេះអ្នកអាចបំបែកបានថា *អំណាចបានផុតកំណត់* ខុសពី *សកម្មភាពដែលបានអនុវត្តបានផ្លាស់ប្តូរ*។


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## សកម្មភាពត្រឹមត្រូវ

ឯកតានៃការយល់ព្រមគឺជា **វត្ថុសកម្មភាព canonical** — មិនមែនជាប្រភេទស្លាកសញ្ញាដូចជា "យល់ព្រមការសងប្រាក់វិញ" ទេ ប៉ុន្តែជាសកម្មភាពដែលកំណត់យ៉ាងពេញលេញនិងច្បាស់លាស់។ ការចុះហត្ថលេខាលើវត្ថុទាំងមូល (និងបង្កើត digest មួយចេញពីវា) គឺជារឿងដែលផ្ដល់ឱ្យយើងអាចបញ្ជាក់ពេលក្រោយថាមនុស្សបានយល់ព្រម *នេះ* ហើយមិនមែនអ្វីផ្សេងទេ។


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ខ្ទង់មួយ អាជ្ញាធរស രണ്ട്

ចំណីរាល់បញ្ជាក់គឺជាខ្ទង់មួយនៃមេរៀន៖ ជាpayloadស្តុកមួយដែលមានវាល `type`, បូករួមជាមួយវត្ថុ `signature` (`alg`, `sig`, `key_id`) ដែល **មិនមែនជា**ផ្នែកនៃបៃតដែលបានចុះហត្ថលេខា។ `verify_envelope` គឺជា ការត្រួតពិនិត្យរចនាសម្ព័ន្ធរួម និងហត្ថលេខាសម្រាប់ប្រភេទបញ្ជាក់ទាំងពីរ; ដែល **ការចុះបញ្ជីកូនសោ pinned** ដែលវាត្រួតពិនិត្យ `signature.key_id` ទៅប្រឆាំង ជាគ្រឿងដែលរក្សាអាជ្ញាធរឱ្យនៅជាផ្ទាល់:

- **បង្កាន់ដើមអនុញ្ញាតិ** (`human.approval.v1`) — អ្នកអនុញ្ញាតមានឈ្មោះ, សកម្មភាព canonical ពេញលេញ ** និង digest របស់វា**, `policy_version`, ម៉ោងចេញនិងផុតកំណត់។ ការបរិភោគម្តងតែម្តងត្រូវបានតាមដាននៅចំណុចខ្សែសង្វាក់។
- **បង្កាន់ដើមសកម្មភាព** (`agent.action.v1`) — អត្តសញ្ញាណភ្នាក់ងារ, `run_id`, digest សកម្មភាព canonical ដូចគ្នា **, លទ្ធផលការប្រតិបត្តិ និងម៉ោង, និង `parent_approval_ref`: `receipt_hash` នៃការអនុញ្ញាតិ, គ្នានៅក្នុងគោលការណ៍ដូចជា `previous_receipt_hash` ក្នុងខ្សែរបស់មេរៀន។

វាល `action_digest` រួមគ្នាជាភ្ជាប់ដែលការចងខ្សែពឹងផ្អែកលើវា។ `key_id` មាននៅក្នុងវត្ថុ signature ជាទំនាក់ទំនងសម្រាប់ស្វែងរកតែប៉ុណ្ណោះ: ប្តូរទិសទៅកាន់កូនសោ pinned ផ្សេងទៀតធ្វើអោយការត្រួតពិនិត្យហត្ថលេខាមិនជោគជ័យ សូម្បីតែមិនផ្តល់អ្វីទេ។


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: ដែលកំណត់ការចងខ្សែត្រូវបានសម្រេចនៅពេលពិត

`verify_chain` មិនមែនជាដុំកូដងាយស្រួលសម្រាប់ការត្រួតពិនិត្យអនុស្សរណៈពីរកន្លែងទេ។ វាជាកន្លែងតែមួយដែលធ្វើការត្រួតពិនិត្យសេចក្តីពិត `action_digest` ដែលបានចែករំលែកយ៉ាងត្រឹមត្រូវ និងគោលនយោបាយ/កូនសោ/កំណត់ពេលផុតកំណត់ **ភាពថ្មី** នៃការអនុញ្ញាត និងការប្រើប្រាស់ **ម្ដងតែមួយ** នៃការអនុញ្ញាត ដោយទាំងអស់ត្រូវបានពិនិត្យប្រឆាំងនឹងសកម្មភាពកំពុងត្រូវបានអនុវត្ត *ឥឡូវនេះ*។

ការបរាជ័យនីមួយៗបដិសេធដោយ **ហេតុផល ផ្សេងគ្នា** ដូច្នេះអ្នកអានអាចដឹងថាអំណាចត្រូវបានបញ្ឈប់ដោយសារ (គោលនយោបាយបានផ្លាស់ប្តូរ, កូនសោបានបង្វិល, កាលបរិច្ឆេទអនុញ្ញាតផុតកំណត់, ការអនុញ្ញាតត្រូវបានប្រើរួចហើយ) ឬសកម្មភាពបានប្រតិបត្តិបានផ្លាស់ប្ដូរចេញពីក្រោមការអនុញ្ញាតដែលនៅតែមានសុពលភាព (ការប្ដូរដោយសារ digest)។


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## អ្វីដែលការចងភ្ជាប់រកឃើញ

រាល់ករណីខាងក្រោមបរាជ័យ **បិទ** ជាមួយនឹង **មូលហេតុខុសប្លែក** មួយ។ ប្លុកដំបូងគឺជាការកំណត់បែបចាស់ (ល្បួងប្រើប្រាស់, អ្នកតំណាងច្របូកច្របល់, ការបញ្ជូនឡើងវិញ, ការបោកប្រាស់នៅលើការ​អនុញ្ញាតណាមួយ, បញ្ចូលព័ត៌មានខុសប្លែត)។ ប្លុកទីពីរជាគូដែលធ្វើឱ្យញហទិន្នបត្ថម្ភដែលពិតជាជាចំណុចពិតនោះ មិនមែនគ្រាន់តែបញ្ជាក់ទេ៖

- **អំណាចចាស់** — គំរូហត្ថលេខានៅតែមានសុពលភាព ប៉ុន្តែកំណែគោលការណ៍បានផ្លាស់ប្តូរ ពាក្យ័រដែលអនុញ្ញាតបានបដិសេធចេញពីឃ្លាំងដែលបានតំរូវ, ឬការអនុញ្ញាតបានផុតកំណត់មុនពេលអនុវត្ត;
- **ការជំនួស digest** — វិញ្ញាបនបត្រប្រតិបត្តិការដែលបានហត្ថលេខាដោយមានសុពលភាព ដែល `parent_approval_ref` សង្កត់ទៅកាន់ការអនុញ្ញាតពិតមួយ ក៏ប៉ុន្តែ digest សកម្មភាព canonical នៃការអនុញ្ញាតនោះ មិនត្រូវគ្នាជាមួយសកម្មភាពដែលកំពុងត្រូវអនុវត្តក្នុងពេលនេះទេ។


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## អ្វីដែលនេះបញ្ជាក់ — ហើយអ្វីដែលវាមិនបញ្ជាក់

**បញ្ជាក់៖** មនុស្សដែលមានឈ្មោះបានអនុម័ត *សកម្មភាពគោលការណ៍ជាក់លាក់នេះ* (សកម្មភាពពេញលេញ + សេចក្តីសង្ខេប ទុកឈ្មោះជាមួយកិច្ចសញ្ញាចេញពីកូនសោដែលបានដោះសោពីកំណត់ត្រាបិទផ្សែង), ហើយភ្នាក់ងារបានអនុវត្ត *ពេញលេញនឹងសកម្មភាពដែលបានអនុម័តនោះ* (សេចក្តីសង្ខេបដូចគ្នា ការទទួលការប្រគល់សេចក្តីសង្ខេបភ្ជាប់ជាមួយការអនុម័តដោយ `receipt_hash`, នីតិវិធីខ្សែស្រឡាយរបស់មេរៀន) — ខណៈដែលកំណែគោលការណ៍ អ្នកដោះសោ និងការបញ្ចប់កំណត់នៅមិនទាន់ផុតកំណត់ នៅពេលតែម្ដង។ ប្រសិនបើផ្នែកណាមួយផ្លាស់ប្តូរ ខ្សែស្រឡាយនោះនឹងបរាជ័យដោយបិទ ហើយហេតុផលទទួលបញ្ញាអោយអ្នកដឹងថា **លក្ខណៈអ្វីមួយ** បានខូចខាត៖ អាជ្ញាបណ្ណចាស់ទាស់ប្រឆាំងនឹងសកម្មភាពដែលបានផ្លាស់ប្តូរ។

**មិនបញ្ជាក់៖** ថាUIនៃការអនុម័តបានបង្ហាញមនុស្ស អ្វីដែលពួកគេគិតថាពួកគេលេខាអង់ក្នុងការចុះហត្ថលេខា (WYSIWYS គឺជាបញ្ហាផ្ទាល់ខ្លួន), ថាគន្លងអត្ថសញ្ញាគឺមិនត្រូវបានបង្ខំឬលួចពីមុនការប្រែប្រួល, ឬថាផលប៉ះពាល់ក្រោមបានស្របទៅនឹងសកម្មភាព។ កិច្ចសញ្ញាដែលបានចុះហត្ថលេខា ≠ បានអនុញ្ញាត៖ កិច្ចសញ្ញាសុពលលើគោលការណ៍ចាស់, គន្លងដែលបានប្ដូរ, កញ្ចក់ផុតកំណត់, ឬសេចក្តីសង្ខេបផ្សេងទៀតមិនផ្តល់អ្វីនៅទីនេះទេ។

ប្រភេទការទទួលទាំងពីរចែករំលែកថង់សំបុត្រមេរៀន និងផ្លូវកូដ `verify_chain` មួយដែលគោលបំណង៖ ការភ្ជាប់ដែលអ្នកបានបង្កើតសម្រាប់ការទទួលសកម្មភាពនៅក្នុងសៀវភៅកំណត់ឈ្មោះគឺជាកូដដដែលដែលពិនិត្យការអនុម័តរបស់មនុស្ស។ កុងត្រាពិនិត្យមួយ អាជ្ញាធរបិទផ្សែងបំបែកពីគ្នា ត្រូវបានភ្ជាប់ដោយសេចក្តីសង្ខេបសកម្មភាពគោលការណ៍ ហើយមិនមានអ្វីផ្សេងទៀត។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
